# mva_02-データの種類と標準化

---

## **要点まとめ**
**Part I：座標と情報**

多変量解析の基礎として、解析対象のデータの性質（尺度）を整理し、数理最適化の土台となる「公平な座標空間」を構築するための幾何学的変換（標準化）を学習します。

### **1. データの種類（4つの尺度）**

解析手法を適用する前に、変数がどのような性質（尺度）を持っているかを確認する必要があります。尺度の種類によって、適用可能な統計演算（平均ベクトルや分散共分散行列の算出など）が限定されるためです。また、これらは質的データを数値に置き換えて分析する「数量化」を考える上での基礎となります。

| 分類 | 尺度 | 特徴 | 統計演算の妥当性 |
| ----- | ----- | ----- | ----- |
| **質的データ** | **名義尺度** | 性別、血液型など。分類のみを目的とし、順序に意味はない。 | 平均・分散は計算不可 |
|  | **順序尺度** | 満足度、順位など. 順序には意味があるが、間隔は一定ではない。 | 平均値の算出には注意が必要 |
| **量的データ** | **間隔尺度** | 温度(℃)、西暦など。数値の差（加減）に意味がある。 | **平均・分散の計算が可能（標準化対象）** |
|  | **比率尺度** | 身長、金額、点数など。絶対的なゼロ（原点）が存在する。 | **平均・分散の計算が可能（標準化対象）** |

### **2. 標準化（Standardization）とその数理的意義**

多変量解析の多くは、目的関数 $J$（予測誤差の最小化や情報量の最大化など）を定義し、それを最適化する問題を解きます。この際、各変数の単位（スケール）や分散が異なると、特定の変数が最適化結果を支配してしまいます。

標準化は、幾何学的には 「**平行移動（中心化）**」 と 「**伸縮（スケーリング）**」 を組み合わせた一次変換であり、以下の状態を実現します。

- **無次元化**: 単位の影響を排除し、異なる変数間での公平な比較を可能にする。
- **寄与度の等質化**: 各変数の分散を 1 に揃え、最適化における各軸の感度を統一する。
- **重心の固定**: データの中心を座標空間の原点 $\mathbf{0}$ に配置する。

---

## **Python 演習課題**

### **mva_02-A**
✅ **目的**: 標準化という一次変換によって、単位や桁数が異なるデータの「歪んだ空間」が、原点を中心とした「公平な空間（各軸のばらつきが 1 の空間）」へどのように再構築されるかを視覚的に理解しましょう。

#### 🖊 **【数理理解】標準化の幾何学的定義**

サンプル数 $n$, 変数数 $d$ のデータ行列 $X \in \mathbb{R}^{n \times d}$ に対する変換を考えます。

##### **1. 平均ベクトルと中心化**
全成分が $1$ のベクトル $\mathbf{1}_n \in \mathbb{R}^n$ を用いて、平均ベクトル $\bar{\mathbf{x}} \in \mathbb{R}^d$ は以下のように定義されます。

$$\bar{\mathbf{x}} = \frac{1}{n} X^{\mathrm{T}} \mathbf{1}_n$$

これを用いて、データの重心を原点に移動させる中心化行列 $X_c$ を求めます。

$$X_c = X - \mathbf{1}_n \bar{\mathbf{x}}^{\mathrm{T}}$$

##### **2. スケーリング行列による伸縮**
各変数の標準偏差 $s_j$ を対角成分に持つ行列 $S = \mathrm{diag}(s_1, s_2, \dots, s_d)$ を用いて、標準化後の行列 $Z$ は逆行列 $S^{-1}$ を右から掛ける一次変換として記述できます。

$$Z = X_c S^{-1}$$

ここで、$S^{-1} = \mathrm{diag}(1/s_1, 1/s_2, \dots, 1/s_d)$ です。標準偏差 $s_j$ が大きい（分布が広い）変数ほど、スケーリング係数 $1/s_j$ は小さくなり、その軸方向のデータは中心に向かって強く圧縮される ことがわかります。

##### **3. 距離への影響**
元のデータ空間におけるユークリッド距離は、数値の桁数や分散が大きい変数の差に支配されます。標準化によって全変数が対等なスケール（平均 0, 分散 1）を持つことで、特定の単位に依存しない、各変数の寄与が等質化された公平な距離計算が可能になります。

#### **具体例で確認してみよう！**
##### **1. 中心化（平行移動）による重心の移動**

まず、各データ $\mathbf{x}_i$ から平均ベクトル $\bar{\mathbf{x}}$ を引く「中心化」のみを実行します。ヒストグラムを重ねて表示することで、分布の形（広がり）は維持したまま、山の頂上が原点 (0) へスライドする様子を確認します。

読み込み先URL:
[https://docs.google.com/spreadsheets/d/1v_FgNUblx_DpXIzmvf9pl1F_eCqMzu10XVjL7YgTfUQ](https://docs.google.com/spreadsheets/d/1v_FgNUblx_DpXIzmvf9pl1F_eCqMzu10XVjL7YgTfUQ)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. データの読み込み
sheet_url = "https://docs.google.com/spreadsheets/d/1v_FgNUblx_DpXIzmvf9pl1F_eCqMzu10XVjL7YgTfUQ/export?format=csv"

df = pd.read_csv(sheet_url, index_col=0)
display(df.head())
X = df.to_numpy()

# 2. 中心化 (Centering) の計算
mean_vec = np.mean(X, axis=0)
X_c = X - mean_vec                      #重心を (0, 0) に移動
n, d = X_c.shape

print(f"\n中心化行列 X_c の形状: (n={n}, d={d})")

In [ ]:
# 3. 中心化によるヒストグラムの平行移動を可視化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# --- 変数1 (x): 中心化の前後比較 ---
sns.histplot(X[:, 0], ax=ax1, color='blue', alpha=0.3, label='Raw', kde=True)
sns.histplot(X_c[:, 0], ax=ax1, color='blue', alpha=0.8, label='Centered (mean=0)', kde=True)
ax1.axvline(0, color='red', linestyle='--')
ax1.set_title("Centering: x1 Distribution")
ax1.legend()

# --- 変数2 (y): 中心化の前後比較 ---
sns.histplot(X[:, 1], ax=ax2, color='green', alpha=0.3, label='Raw', kde=True)
sns.histplot(X_c[:, 1], ax=ax2, color='green', alpha=0.8, label='Centered (mean=0)', kde=True)
ax2.axvline(0, color='red', linestyle='--')
ax2.set_title("Centering: x2 Distribution")
ax2.legend()

plt.tight_layout()
plt.show()

##### **2. スケーリング（伸縮）による空間の再構築**

次に、中心化したデータを標準偏差で割り、すべての軸のばらつきを 1 に揃えます。左図には元のスケールでの分布を、右図には標準化してズームした分布を並べて表示し、各変数の $1\sigma$ の範囲を長方形で描画することで、空間がどのように整理されたかを確認します。

In [ ]:
import matplotlib.patches as patches

# 4. 標準偏差によるスケーリング (Scaling)
std_vec = np.std(X, axis=0)
S_inv = np.diag(1.0 / std_vec)
Z = X_c @ S_inv                         # 標準化一次変換 (Z = X_c S^-1)

# 5. 座標空間の比較 (元のデータ空間 vs 統一された座標空間)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# --- 左図: 標準化前 (単位の違いにより空間が歪んでいる) ---
ax1.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolors='k')
rect1 = patches.Rectangle((mean_vec[0]-std_vec[0], mean_vec[1]-std_vec[1]),
                         2*std_vec[0], 2*std_vec[1], linewidth=2, edgecolor='red', facecolor='none', label='1-sigma range')
ax1.add_patch(rect1)
ax1.set_xlim([-100, 1000]); ax1.set_ylim([-100, 1000])
ax1.set_aspect('equal')
ax1.set_title(f"Original Space\n  std(x1) ={std_vec[0]:.0f} vs std(x2) ={std_vec[1]:.0f}")
ax1.set_xlabel("x1"); ax1.set_ylabel("x2")
ax1.grid(True); ax1.legend()

# --- 右図: 標準化後 (各軸のばらつきが 1 に揃った公平な空間) ---
ax2.scatter(Z[:, 0], Z[:, 1], alpha=0.8, color='orange', edgecolors='k')
rect2 = patches.Rectangle((-1, -1), 2, 2, linewidth=2, edgecolor='red', facecolor='none', label='Standardized 1-sigma')
ax2.add_patch(rect2)
ax2.axhline(0, color='black', lw=1); ax2.axvline(0, color='black', lw=1)
ax2.set_xlim([-4, 4]); ax2.set_ylim([-4, 4])
ax2.set_aspect('equal')
ax2.set_title("Standardized Space")
ax2.set_xlabel("z-Score (x1)"); ax2.set_ylabel("z-Score (x2)")
ax2.grid(True); ax2.legend()

plt.tight_layout()
plt.show()

### **mva_02-B**
✅ **目的**: ``scikit-learn`` の ``StandardScaler`` を使い、大規模なデータ行列を一括で公平な座標系へ変換する手順を習得しましょう。

#### 💻 **【実装・応用】 ``StandardScaler`` による実務的な前処理**

実際の分析では、ライブラリを用いて多数の変数を一括で処理します。``fit_transform`` メソッドが内部で平均ベクトルと標準偏差行列 $S$ を算出し、行列演算 $Z = (X - \mathbf{1}\bar{\mathbf{x}}^{\mathrm{T}}) S^{-1}$ を実行していることを意識してください。

詳細な仕様については、以下の公式ドキュメントを参照してください。

``scikit-learn`` 公式ドキュメント: [``StandardScaler``](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

#### **具体例で確認してみよう！**
第1回で使用したスプレッドシートデータ（数学・英語の点数）を対象に、ライブラリによる一次変換を適用します。

読み込み先URL:
[https://docs.google.com/spreadsheets/d/1ulRrXvp8IaHRLOFcN3a6u2D1OzxANiBCmLgHmTyevgs](https://docs.google.com/spreadsheets/d/1ulRrXvp8IaHRLOFcN3a6u2D1OzxANiBCmLgHmTyevgs)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. 表データの読み込み (前回使用したGoogleスプレッドシート)
sheet_url = "https://docs.google.com/spreadsheets/d/1ulRrXvp8IaHRLOFcN3a6u2D1OzxANiBCmLgHmTyevgs/export?format=csv"
df = pd.read_csv(sheet_url, index_col=0)

print("--- 標準化前のデータフレーム (Raw Matrix X) ---")
display(df)

In [ ]:
# 2. StandardScaler (一次変換器) の適用
scaler = StandardScaler()

# 数値データを抽出し、一括で行列演算を実行
# fitで平均・標準偏差を学習し、transformで変換を行う
scaled_data = scaler.fit_transform(df)

# 3. 結果をラベル付きのデータフレームに戻す
df_scaled = pd.DataFrame(scaled_data, index=df.index, columns=df.columns)

print("\n--- 標準化後のデータフレーム (Standardized Matrix Z) ---")
display(df_scaled)

In [ ]:
# 4. 統計量の確認
# 全ての軸において平均 0、標準偏差 1 となり、空間が整理されたことを確認します
print("\n平均ベクトル (axis=0):")
print(df_scaled.mean(axis=0, numeric_only=True).round(10))

print("\n標準偏差ベクトル (axis=0, ddof=0):")
print(df_scaled.std(axis=0, ddof=0)) # ddof: Delta Degree of Freedom